# Hands-On LangChain #10: ChatBots Memory

Companion to [the complete article](https://todatabeyond.com/blog/hands-on-langchain-for-llms-app-chatbots-memory).

The default path executes real legacy memory containers, actual CPU FLAN-T5-small conversation/summary inference, a real tokenizer, and current session-history composition without API keys. First use downloads the public model; outputs can be inaccurate. Scripted summary and session fixtures separately verify integration behavior and are explicitly NOT inference. All 22 maintained original hosted steps are preserved in a disabled appendix. Historical article outputs are not claimed as reproduced.

The source uses deprecated ConversationChain and memory classes, retained via langchain-classic only to explain the archive. For new agent applications use current LangChain short-term memory with checkpointing; the LangGraph example below demonstrates thread isolation, not a production persistence/authentication system. Memory is application-managed context, not a change to model weights.

## 1. Install dependencies

In [ ]:
!pip install -q "langchain-core>=1.0,<2" "langchain-classic>=1.0,<2" "langchain-openai>=1.0,<2" "langgraph>=1.0,<2" "transformers>=4.57,<5" torch sentencepiece tiktoken python-dotenv

## 2. Real local model and tokenizer
All local counts use the FLAN-T5 tokenizer and are not OpenAI billable-token estimates. The demonstration rejects prompts beyond its 512-token input budget. Small-model responses are printed with repr and evaluated as observations, not certified answers.

In [ ]:
from pathlib import Path
import tempfile
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.language_models.llms import LLM

torch.set_num_threads(2)
cache = str(Path(tempfile.gettempdir()) / "tdb-flan-t5-cache")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small", cache_dir=cache)
answer_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small", cache_dir=cache)
answer_model.eval()
calls = []

class LocalAnswerLLM(LLM):
    @property
    def _llm_type(self): return "local-flan-t5-small"
    def get_num_tokens(self, text): return len(tokenizer.encode(text))
    def _call(self, prompt, stop=None, run_manager=None, **kwargs):
        inputs = tokenizer(prompt, return_tensors="pt", truncation=False)
        if inputs["input_ids"].shape[1] > 512:
            raise ValueError("Prompt exceeds this demo's 512-token input budget.")
        with torch.inference_mode():
            tokens = answer_model.generate(**inputs, max_new_tokens=64, do_sample=False)
        answer = tokenizer.decode(tokens[0], skip_special_tokens=True)
        calls.append({"prompt": prompt, "answer": answer})
        return answer

llm = LocalAnswerLLM()
print("Model/tokenizer: google/flan-t5-small (CPU, no API key)")

## 3. Full buffer: real container and actual model calls

In [ ]:
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import (
    ConversationBufferMemory, ConversationBufferWindowMemory,
    ConversationTokenBufferMemory, ConversationSummaryBufferMemory,
)
full_memory = ConversationBufferMemory()
full_chain = ConversationChain(llm=llm, memory=full_memory, verbose=False)
questions = ["Hi, my name is Youssef", "What is 1+1?", "What is my name?"]
calls.clear()
full_answers = [full_chain.invoke({"input": q})["response"] for q in questions]
assert len(calls) == 3 and len(full_memory.chat_memory.messages) == 6
assert "Human: Hi, my name is Youssef" in calls[-1]["prompt"]
for q, a in zip(questions, full_answers):print("Full-buffer local answer:", q, "→", repr(a))
print("Full buffer: 6 messages; initial human turn reached the third prompt")
assert full_memory.load_memory_variables({})["history"] == full_memory.buffer
full_memory.clear()
assert full_memory.load_memory_variables({}) == {"history": ""}
print("Full buffer clear: verified")

## 4. Window: visible context versus underlying storage
The legacy k=1 window exposes the latest exchange but may retain all underlying messages. It is not a storage-deletion policy. Recreate the chain with window memory, which the source omitted. Generated replies could repeat earlier facts, so absence of the original human turn is not proof that a name is absent from every answer.

In [ ]:
window_memory = ConversationBufferWindowMemory(k=1)
window_memory.save_context({"input": "Hi"}, {"output": "What's up"})
window_memory.save_context({"input": "Not much, just hanging"}, {"output": "Cool"})
window_history = window_memory.load_memory_variables({})["history"]
assert window_history == "Human: Not much, just hanging\nAI: Cool"
assert len(window_memory.chat_memory.messages) == 4
print("Window k=1 visible fixture:", repr(window_history))
print("Underlying stored fixture messages:", len(window_memory.chat_memory.messages))
window_memory.clear()
assert not window_memory.chat_memory.messages
window_chain = ConversationChain(llm=llm, memory=window_memory, verbose=False)
calls.clear()
window_answers = [window_chain.invoke({"input": q})["response"] for q in questions]
assert len(calls) == 3
assert "Human: Hi, my name is Youssef" not in calls[-1]["prompt"]
assert "Human: What is 1+1?" in calls[-1]["prompt"]
for q, a in zip(questions, window_answers):print("Window local answer:", q, "→", repr(a))
print("Window third prompt: latest exchange included; initial human turn excluded")

## 5. Token buffers: actual tokenizer and pruning
The legacy token buffer prunes individual messages, not necessarily complete human/AI pairs. Its limit concerns saved history, not the whole prompt plus new input and generated response. Counts are model-specific; changing tokenizers changes which messages remain.

In [ ]:
pairs = [("AI is what?!", "Amazing!"), ("Backpropagation is what?", "Beautiful!"), ("Chatbots are what?", "Charming!")]
counts = {}
for budget in (50, 100, 20):
    token_memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=budget)
    for question, answer in pairs:
        token_memory.save_context({"input": question}, {"output": answer})
    count = llm.get_num_tokens_from_messages(token_memory.chat_memory.messages)
    counts[budget] = count
    assert count <= budget
    print("Token budget:", budget, "retained tokens:", count, "history:", repr(token_memory.load_memory_variables({})["history"]))
assert counts[100] >= counts[50] >= counts[20]
assert len(token_memory.chat_memory.messages) < 6
token_memory.clear()
assert token_memory.load_memory_variables({}) == {"history": ""}
print("Token-buffer budget and clear assertions passed")

## 6. Summary buffer: deterministic integration checks

The scripted summary is a fixed fixture, not model inference. It verifies when the real summary memory invokes its LLM and how it combines summary and recent messages. The source accidentally reverses 400/100 limits; the maintained original workflow corrects them. The summary itself is not bounded by max_token_limit, so budget the complete prompt separately.

In [ ]:
# create a long string
schedule = "There is a meeting at 8am with your product team. \
You will need your powerpoint presentation prepared. \
9am-12pm have time to work on your LangChain \
project which will go quickly because Langchain is such a powerful tool. \
At Noon, lunch at the italian resturant with a customer who is driving \
from over an hour away to meet you to understand the latest in AI. \
Be sure to bring your laptop to show the latest LLM demo."

In [ ]:
from langchain_core.language_models.llms import LLM
scripted_calls = []
class ScriptedSummaryLLM(LLM):
    @property
    def _llm_type(self):return "explicit-scripted-summary-fixture"
    def get_num_tokens(self, text):return len(tokenizer.encode(text))
    def _call(self, prompt, stop=None, run_manager=None, **kwargs):
        scripted_calls.append(prompt)
        return "Scripted fixture: product meeting at 8am, project work at 9am, lunch with an AI customer."

scripted_llm = ScriptedSummaryLLM()
summary_memory = ConversationSummaryBufferMemory(llm=scripted_llm, max_token_limit=400)
for q, a in [("Hello", "What's up"), ("Not much, just hanging", "Cool"), ("What is on the schedule today?", schedule)]:
    summary_memory.save_context({"input": q}, {"output": a})
assert not scripted_calls and summary_memory.moving_summary_buffer == ""
assert schedule in summary_memory.load_memory_variables({})["history"]
summary_memory.max_token_limit = 100
summary_memory.prune()
assert len(scripted_calls) == 1 and "8am" in scripted_calls[0]
assert summary_memory.moving_summary_buffer.startswith("Scripted fixture:")
assert llm.get_num_tokens_from_messages(summary_memory.chat_memory.messages) <= 100
assert "System: Scripted fixture:" in summary_memory.load_memory_variables({})["history"]
print("Summary fixture: no call below 400; one call after pruning to 100")
print("Scripted combined history:", repr(summary_memory.load_memory_variables({})["history"]))
summary_memory.clear()
assert summary_memory.moving_summary_buffer == "" and not summary_memory.chat_memory.messages
print("Summary buffer clear: messages and moving summary removed")

## 7. Real local summarization
This is actual FLAN-T5-small inference through the legacy summary memory, not the scripted test above. Its summary may omit, distort, or invent facts. The assertions verify that pruned context reached the summarizer and its captured response became memory—not that the summary is accurate.

In [ ]:
real_summary_memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=400)
for q, a in [("Hello", "What's up"), ("Not much, just hanging", "Cool"), ("What is on the schedule today?", schedule)]:
    real_summary_memory.save_context({"input": q}, {"output": a})
calls.clear()
real_summary_memory.max_token_limit = 100
real_summary_memory.prune()
assert len(calls) == 1 and "8am" in calls[0]["prompt"]
assert calls[0]["answer"] == real_summary_memory.moving_summary_buffer
print("Real local summary:", repr(real_summary_memory.moving_summary_buffer))
print("Real summary retained-buffer tokens:", llm.get_num_tokens_from_messages(real_summary_memory.chat_memory.messages))
print("Summary quality is not certified; inspect against the original schedule.")

## 8. Current LangGraph thread history, isolation, and clearing

LangGraph stores thread state using a checkpointer. The response node here is explicitly scripted, not an LLM. Thread IDs must be authorized per user in a real service. InMemorySaver does not survive process restarts and is not safe multi-tenant infrastructure by itself. Budget prompts and define privacy/deletion policies.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver

thread_inputs = []
def scripted_chat(state: MessagesState):
    thread_inputs.append(list(state["messages"]))
    return {"messages": [AIMessage(content="Scripted response; not model inference.")]}

builder = StateGraph(MessagesState)
builder.add_node("chat", scripted_chat)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)
def thread_config(thread_id):
    return {"configurable": {"thread_id": thread_id}}
def thread_chat(text, thread_id):
    return graph.invoke({"messages": [HumanMessage(content=text)]}, thread_config(thread_id))

thread_chat("My name is Youssef", "alice")
thread_chat("What is my name?", "alice")
assert len(graph.get_state(thread_config("alice")).values["messages"]) == 4
assert any(m.content == "My name is Youssef" for m in thread_inputs[-1])
thread_chat("Hello", "bob")
assert not any(m.content == "My name is Youssef" for m in thread_inputs[-1])
assert len(graph.get_state(thread_config("bob")).values["messages"]) == 2
checkpointer.delete_thread("alice")
assert not graph.get_state(thread_config("alice")).values
assert len(graph.get_state(thread_config("bob")).values["messages"]) == 2
print("Current LangGraph history: follow-up context, thread isolation, and per-thread deletion verified (scripted node)")
print("All memory integration assertions passed.")

## 9. Complete optional hosted original workflow

All 22 original steps follow in source order, with imports/invoke/environment setup updated, window chain reconstruction restored, the full token-buffer limit corrected to 100, and summary limits corrected to 400 then 100. Historical GPT-3.5 naming is not hardcoded as a current default. Set OPENAI_API_KEY and OPENAI_CHAT_MODEL locally and enable the flag to make paid calls. Set temperature only when supported. Hosted token counting must support the chosen model; tokenization may need a first-run download and these counts differ from local FLAN-T5. Hosted calls were not executed. Do not commit credentials, log private conversations via verbose, or treat model summaries as an authoritative record. Legacy classes are for archive compatibility, not recommended new application APIs.

In [ ]:
RUN_HOSTED_EXAMPLE = False

In [ ]:
if RUN_HOSTED_EXAMPLE:
    import os
    from dotenv import load_dotenv
    load_dotenv()
    for name in ("OPENAI_API_KEY", "OPENAI_CHAT_MODEL"):
        if not os.environ.get(name):
            raise RuntimeError(f"Set {name} locally before running hosted examples.")
    model_options = {}
    if os.environ.get("OPENAI_TEMPERATURE") is not None:
        model_options["temperature"] = float(os.environ["OPENAI_TEMPERATURE"])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    llm_model = os.environ["OPENAI_CHAT_MODEL"]

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_openai import ChatOpenAI
    from langchain_classic.chains import ConversationChain
    from langchain_classic.memory import ConversationBufferMemory
    
    llm = ChatOpenAI(model=llm_model, **model_options)
    memory = ConversationBufferMemory()
    conversation = ConversationChain(
        llm=llm, 
        memory = memory,
        verbose=True
    )

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(conversation.invoke({"input": "Hi, my name is Youssef"})["response"])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(conversation.invoke({"input": "What is 1+1?"})["response"])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(conversation.invoke({"input": "What is my name?"})["response"])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(memory.buffer)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(memory.load_memory_variables({}))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    memory = ConversationBufferMemory()
    print(memory.save_context({"input": "Hi"}, 
                    {"output": "What's up"}))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(memory.buffer)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_classic.memory import ConversationBufferWindowMemory
    memory = ConversationBufferWindowMemory(k=1)
    print(memory.load_memory_variables({}))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    memory.save_context({"input": "Hi"},
                        {"output": "What's up"})
    memory.save_context({"input": "Not much, just hanging"},
                        {"output": "Cool"})
    print(memory.load_memory_variables({}))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    # The source omitted rebuilding the chain with window memory.
    memory = ConversationBufferWindowMemory(k=1)
    conversation = ConversationChain(llm=llm, memory=memory, verbose=True)
    print(conversation.invoke({"input": "Hi, my name is Youssef"})["response"])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(conversation.invoke({"input": "What is 1+1?"})["response"])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(conversation.invoke({"input": "What is my name?"})["response"])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_classic.memory import ConversationSummaryBufferMemory
    
    from langchain_classic.memory import ConversationTokenBufferMemory
    llm = ChatOpenAI(model=llm_model, **model_options)
    
    memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=50)
    memory.save_context({"input": "AI is what?!"},
                        {"output": "Amazing!"})
    memory.save_context({"input": "Backpropagation is what?"},
                        {"output": "Beautiful!"})
    memory.save_context({"input": "Chatbots are what?"}, 
                        {"output": "Charming!"})
    
    print(memory.load_memory_variables({}))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=100)
    memory.save_context({"input": "AI is what?!"},
                        {"output": "Amazing!"})
    memory.save_context({"input": "Backpropagation is what?"},
                        {"output": "Beautiful!"})
    memory.save_context({"input": "Chatbots are what?"}, 
                        {"output": "Charming!"})
    print(memory.load_memory_variables({}))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=20)
    memory.save_context({"input": "AI is what?!"},
                        {"output": "Amazing!"})
    memory.save_context({"input": "Backpropagation is what?"},
                        {"output": "Beautiful!"})
    memory.save_context({"input": "Chatbots are what?"}, 
                        {"output": "Charming!"})
    print(memory.load_memory_variables({}))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    # create a long string
    schedule = "There is a meeting at 8am with your product team. \
    You will need your powerpoint presentation prepared. \
    9am-12pm have time to work on your LangChain \
    project which will go quickly because Langchain is such a powerful tool. \
    At Noon, lunch at the italian resturant with a customer who is driving \
    from over an hour away to meet you to understand the latest in AI. \
    Be sure to bring your laptop to show the latest LLM demo."

In [ ]:
if RUN_HOSTED_EXAMPLE:
    memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=400)
    memory.save_context({"input": "Hello"}, {"output": "What's up"})
    memory.save_context({"input": "Not much, just hanging"},
                        {"output": "Cool"})
    memory.save_context({"input": "What is on the schedule today?"}, 
                        {"output": f"{schedule}"})
    
    print(memory.load_memory_variables({}))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100)
    memory.save_context({"input": "Hello"}, {"output": "What's up"})
    memory.save_context({"input": "Not much, just hanging"},
                        {"output": "Cool"})
    memory.save_context({"input": "What is on the schedule today?"}, 
                        {"output": f"{schedule}"})
    print(memory.load_memory_variables({}))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    conversation = ConversationChain(
        llm=llm, 
        memory = memory,
        verbose=True
    )
    print(conversation.invoke({"input": "What would be a good demo to show?"})["response"])